# Milestone 4 — Task 2: Parquet + DuckDB Migration (Jiro)

## Overview

Switch data loading from CSV/pandas to **parquet + DuckDB via ibis**. All filtering
in the Shiny pages should happen at the database level via ibis expressions, and only
materialize to pandas at the rendering boundary (`.to_pandas()`).

**Branch:** `feat/parquet-duckdb`  
**Blocked by:** Task 1 (Environment Setup — `ibis-framework[duckdb]` must already be in `requirements.txt`)  
**Key files you will create/edit:**

| Action | File |
|--------|------|
| **Create** | `data/processed/financial_statement.parquet` |
| **Rewrite** | `src/data.py` |
| **Edit** | `src/pages/sector.py` |
| **Edit** | `src/pages/company.py` |
| **Update** | `tests/test_data.py` |

> `src/pages/ai_explorer.py` does **not** need changes — querychat takes a pandas DataFrame
> and does its own SQL filtering internally.

---

## Step 1: Create the feature branch

```bash
git checkout develop
git pull origin develop
git checkout -b feat/parquet-duckdb
git push origin feat/parquet-duckdb
```

---

## Step 2: Convert the CSV to Parquet

Create `data/processed/` and convert the raw CSV into a cleaned parquet file.
The cleaning (strip column names, uppercase Category) is baked into the parquet
so `data.py` no longer needs to do it at load time.

```bash
mkdir -p data/processed
```

Run this one-time conversion script:

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("../../data/raw/financial_statement.csv")
parquet_path = Path("../../data/processed/financial_statement.parquet")

data = pd.read_csv(csv_path, encoding="utf-8-sig")
data.columns = data.columns.str.strip()
data["Category"] = data["Category"].str.upper()
data.to_parquet(parquet_path, index=False)

print(f"Converted {len(data)} rows, {len(data.columns)} columns")
print(f"Saved to: {parquet_path}")

**Verify** the parquet file loads correctly with ibis:

In [ ]:
import ibis

con = ibis.duckdb.connect()
tbl = con.read_parquet(str(parquet_path))
print(tbl.schema())
print(f"Row count: {tbl.count().execute()}")

**Expected output:**
- 161 rows, 23 columns
- Schema shows `Year` as `int64`, `Company`/`Category` as `string`, financial metrics as `float64`

---

## Step 3: Rewrite `src/data.py`

Replace the CSV-based `load_data()` approach with ibis + DuckDB.

### What changes:

| Before (CSV) | After (Parquet + ibis) |
|---|---|
| `import pandas as pd` | `import ibis` |
| `DATA_PATH = ... / "raw" / "financial_statement.csv"` | `PARQUET_PATH = ... / "processed" / "financial_statement.parquet"` |
| `load_data()` function reads CSV, strips columns, uppercases | No function — ibis reads pre-cleaned parquet directly |
| `df = load_data()` | `con = ibis.duckdb.connect()` → `tbl = con.read_parquet(...)` → `df = tbl.to_pandas()` |
| Year min/max computed from `df` in pages | `YEAR_MIN` / `YEAR_MAX` computed once from ibis |

### New exports:
- `con` — the DuckDB connection
- `tbl` — ibis Table expression (lazy — no data loaded until `.execute()` or `.to_pandas()`)
- `df` — full pandas DataFrame (still needed by querychat and chart builders)
- `YEAR_MIN`, `YEAR_MAX` — integer year range constants

**Replace the entire contents of `src/data.py` with:**

In [ ]:
# ===== NEW src/data.py =====

"""Data loading, cleaning, and shared constants for the fin-health dashboard."""

from pathlib import Path

import ibis

PARQUET_PATH = Path(__file__).parent.parent / "data" / "processed" / "financial_statement.parquet"

# --- ibis / DuckDB connection ---
con = ibis.duckdb.connect()
tbl = con.read_parquet(str(PARQUET_PATH))

# Full pandas DataFrame (used by querychat and chart builders)
df = tbl.to_pandas()

# Year range (computed once from ibis, avoids scanning pandas)
YEAR_MIN = int(tbl["Year"].min().execute())
YEAR_MAX = int(tbl["Year"].max().execute())

CATEGORY_COMPANIES = {
    "BANK": ["AIG", "BCS"],
    "ELEC": ["INTC", "NVDA"],
    "FINANCE": ["SHLDQ"],
    "FINTECH": ["PYPL"],
    "FOOD": ["MCD"],
    "IT": ["AAPL", "GOOG", "MSFT"],
    "LOGI": ["AMZN"],
    "MANUFACTURING": ["PCG"],
}
ALL_SECTORS = sorted(CATEGORY_COMPANIES.keys())

METRIC_CHOICES = {
    "Net Profit Margin": "%",
    "ROE": "%",
    "ROA": "%",
    "ROI": "%",
    "Revenue": "USD",
    "Net Income": "USD",
    "EBITDA": "USD",
    "Current Ratio": "",
    "Debt/Equity Ratio": "",
}

### Key design decisions:

1. **`ibis.duckdb.connect()`** — in-memory DuckDB, no file needed. The parquet IS the storage.
2. **`tbl`** is lazy — no data is loaded until you call `.execute()` or `.to_pandas()`. Pages will filter `tbl` via ibis expressions, then materialize only the filtered subset.
3. **`df = tbl.to_pandas()`** — we still need a full pandas DataFrame because querychat and all chart builder functions expect pandas. This is loaded once at startup.
4. **`YEAR_MIN` / `YEAR_MAX`** — computed once via ibis scalar aggregation. Pages import these instead of calling `int(df["Year"].min())` repeatedly.
5. **`load_data()` function removed** — no longer needed since parquet is pre-cleaned.

---

## Step 4: Update `src/pages/sector.py`

Wire all filtering through ibis expressions in `@reactive.calc`.

### Changes summary:

| What | Before | After |
|------|--------|-------|
| Import | `from data import ALL_SECTORS, METRIC_CHOICES, df` | `from data import ALL_SECTORS, METRIC_CHOICES, YEAR_MIN, YEAR_MAX, tbl` |
| Year slider values | `int(df["Year"].min())`, `int(df["Year"].max())` | `YEAR_MIN`, `YEAR_MAX` |
| Reset handler | `int(df["Year"].min())`, `int(df["Year"].max())` | `YEAR_MIN`, `YEAR_MAX` |
| `p1_filtered_data()` | pandas boolean indexing on `df` | `tbl.filter(...)` → `.to_pandas()` |

### Detailed change 1: Import line

**Find:**
```python
from data import ALL_SECTORS, METRIC_CHOICES, df
```

**Replace with:**
```python
from data import ALL_SECTORS, METRIC_CHOICES, YEAR_MIN, YEAR_MAX, tbl
```

---

### Detailed change 2: Year slider in `sector_ui()`

**Find:**
```python
    year_slider = ui.input_slider(
        id="p1_year_range",
        label="Period",
        min=int(df["Year"].min()),
        max=int(df["Year"].max()),
        value=[int(df["Year"].min()), int(df["Year"].max())],
        sep="",
    )
```

**Replace with:**
```python
    year_slider = ui.input_slider(
        id="p1_year_range",
        label="Period",
        min=YEAR_MIN,
        max=YEAR_MAX,
        value=[YEAR_MIN, YEAR_MAX],
        sep="",
    )
```

---

### Detailed change 3: Reset handler in `sector_server()`

**Find:**
```python
        ui.update_slider(
            "p1_year_range", value=[int(df["Year"].min()), int(df["Year"].max())]
        )
```

**Replace with:**
```python
        ui.update_slider("p1_year_range", value=[YEAR_MIN, YEAR_MAX])
```

---

### Detailed change 4: `p1_filtered_data()` — the core migration

This is the most important change. Replace pandas boolean indexing with ibis `.filter()` expressions.

**Find:**
```python
    @reactive.calc
    def p1_filtered_data():
        """Filter dataset by selected year range and sector."""
        year_min, year_max = input.p1_year_range()
        sector = input.p1_sector()
        filtered = df[(df["Year"] >= year_min) & (df["Year"] <= year_max)]
        if sector and "All" not in sector:
            filtered = filtered[filtered["Category"].isin(sector)]
        return filtered
```

**Replace with:**
```python
    @reactive.calc
    def p1_filtered_data():
        """Filter dataset via ibis expressions, then materialize to pandas."""
        year_min, year_max = input.p1_year_range()
        sector = input.p1_sector()
        expr = tbl.filter(tbl["Year"] >= year_min, tbl["Year"] <= year_max)
        if sector and "All" not in sector:
            expr = expr.filter(tbl["Category"].isin(sector))
        return expr.to_pandas()
```

**How ibis filtering works:**
- `tbl.filter(condition1, condition2)` — multiple conditions are AND-ed together
- `tbl["Column"]` — references a column (returns an ibis expression, not data)
- `.isin(list)` — same semantics as pandas `.isin()`
- `.to_pandas()` — materializes the filtered result into a pandas DataFrame
- The SQL is pushed down to DuckDB — only matching rows are returned

> **Everything downstream** (KPIs, charts, tables) still receives a pandas DataFrame,
> so no changes needed in the rendering code.

---

### Complete new `src/pages/sector.py`

Here is the complete file after all edits. Use this as a reference to verify your changes:

In [ ]:
# ===== NEW src/pages/sector.py =====

"""Page 1: Sector Analysis — UI layout and server logic."""

import pandas as pd
from shiny import reactive, render, ui
from shinywidgets import output_widget, render_altair
from data import ALL_SECTORS, METRIC_CHOICES, YEAR_MIN, YEAR_MAX, tbl
from components.kpi_card import kpi_card
from charts.altair_charts import (
    build_metric_trend,
    build_peer_scatter,
    build_sector_bar,
)


def sector_ui():
    """Return the full Page 1 layout (sidebar + KPI row + chart row + peer row)."""
    # Sidebar inputs
    year_slider = ui.input_slider(
        id="p1_year_range",
        label="Period",
        min=YEAR_MIN,
        max=YEAR_MAX,
        value=[YEAR_MIN, YEAR_MAX],
        sep="",
    )
    sector_select = ui.input_selectize(
        id="p1_sector",
        label="Sector",
        choices=["All"] + ALL_SECTORS,
        selected="All",
        multiple=True,
    )
    metric_select = ui.input_selectize(
        id="p1_metric",
        label="Metric",
        choices=list(METRIC_CHOICES.keys()),
        selected="Net Profit Margin",
    )
    reset = ui.input_action_button("p1_reset", "Reset Filters")
    sidebar = ui.sidebar(
        ui.h4("Analytics Filters"),
        year_slider,
        sector_select,
        metric_select,
        reset,
        open="desktop",
    )

    # KPI cards — using the reusable kpi_card() factory
    card_avg_margin = kpi_card(
        header="Avg Profit Margin",
        value_id="p1_avg_margin",
        trend_id="p1_margin_trend",
        label_id="p1_margin_badge",
    )
    card_top_sector = kpi_card(
        header="Top Sector",
        value_id="p1_top_sector",
        label_id="p1_index_performance_display",
    )
    card_revenue_growth = kpi_card(
        header="Revenue Growth",
        value_id="p1_revenue_growth_value",
        trend_id="p1_revenue_trend",
        label_id="p1_revenue_growth_label",
    )
    kpi_row = ui.div(
        ui.layout_columns(
            card_avg_margin,
            card_top_sector,
            card_revenue_growth,
            col_widths=[4, 4, 4],
        ),
        class_="kpi-card-row",
    )

    # Chart cards
    card_sector_profitability = ui.card(
        ui.card_header("Sector Comparison"),
        output_widget("p1_chart_a"),
        full_screen=True,
    )
    card_trend = ui.card(
        ui.card_header("Historical Trend"),
        output_widget("p1_chart_b"),
        full_screen=True,
    )
    chart_row = ui.layout_columns(
        card_sector_profitability,
        card_trend,
        col_widths=[6, 6],
    )

    # Peer + Table cards
    card_peer = ui.card(
        ui.card_header("Peer Benchmarking"),
        output_widget("p1_chart_c"),
        full_screen=True,
    )
    card_details = ui.card(
        ui.card_header("Company Details"),
        ui.output_data_frame("p1_table_d"),
    )
    peer_row = ui.layout_columns(
        card_peer,
        card_details,
        col_widths=[6, 6],
    )

    return ui.layout_sidebar(
        sidebar,
        ui.page_fillable(
            ui.h2("US Corporate Profitability Analytics"),
            kpi_row,
            chart_row,
            peer_row,
        ),
    )


def sector_server(input, output, session):
    """All Page 1 reactive calcs and renderers."""

    @reactive.calc
    def p1_selected_metric():
        """Return the selected metric, falling back to default if cleared."""
        metric = input.p1_metric()
        if not metric or metric not in METRIC_CHOICES:
            return "Net Profit Margin"
        return metric

    @reactive.effect
    @reactive.event(input.p1_reset)
    def _():
        ui.update_slider("p1_year_range", value=[YEAR_MIN, YEAR_MAX])
        ui.update_selectize("p1_sector", selected="All")
        ui.update_select("p1_metric", selected="Net Profit Margin")

    @reactive.calc
    def p1_filtered_data():
        """Filter dataset via ibis expressions, then materialize to pandas."""
        year_min, year_max = input.p1_year_range()
        sector = input.p1_sector()
        expr = tbl.filter(tbl["Year"] >= year_min, tbl["Year"] <= year_max)
        if sector and "All" not in sector:
            expr = expr.filter(tbl["Category"].isin(sector))
        return expr.to_pandas()

    # ... rest of server logic remains UNCHANGED ...
    # All KPIs, charts, and tables still work because p1_filtered_data()
    # returns a pandas DataFrame — same interface as before.

> **Note:** Only the import line, `sector_ui()` slider values, the reset handler, and `p1_filtered_data()` change.
> All KPI outputs, chart renderers, and the data table remain untouched because they still
> receive a pandas DataFrame from `p1_filtered_data()`.

---

## Step 5: Update `src/pages/company.py`

### Changes summary:

| What | Before | After |
|------|--------|-------|
| Import | `from data import ALL_SECTORS, CATEGORY_COMPANIES, df` | `from data import ALL_SECTORS, CATEGORY_COMPANIES, YEAR_MIN, YEAR_MAX, tbl` |
| `p2_year_slider` | `df[df["Company"] == company]` pandas filter | `tbl.filter(tbl["Company"] == company).to_pandas()` |
| Year fallback | `int(df["Year"].min())` | `YEAR_MIN` / `YEAR_MAX` |
| `p2_filtered_data()` | pandas boolean indexing | `tbl.filter(...)` → `.to_pandas()` |
| Company trend charts | `df[df["Company"] == company]` in each renderer | New `p2_company_data()` reactive calc via ibis |

---

### Detailed change 1: Import line

**Find:**
```python
from data import ALL_SECTORS, CATEGORY_COMPANIES, df
```

**Replace with:**
```python
from data import ALL_SECTORS, CATEGORY_COMPANIES, YEAR_MIN, YEAR_MAX, tbl
```

---

### Detailed change 2: `p2_year_slider` reactive UI

**Find:**
```python
    @render.ui
    def p2_year_slider():
        company = input.company()
        company_data = df[df["Company"] == company]
        if company_data.empty:
            year_min = int(df["Year"].min())
            year_max = int(df["Year"].max())
```

**Replace with:**
```python
    @render.ui
    def p2_year_slider():
        company = input.company()
        company_expr = tbl.filter(tbl["Company"] == company)
        company_data = company_expr.to_pandas()
        if company_data.empty:
            year_min = YEAR_MIN
            year_max = YEAR_MAX
```

The rest of `p2_year_slider` stays the same (it already works with `company_data` as pandas).

---

### Detailed change 3: `p2_filtered_data()` — core migration

**Find:**
```python
    @reactive.calc
    def p2_filtered_data():
        category = input.category()
        company = input.company()
        year = input.year()
        return df[
            (df["Category"] == category)
            & (df["Company"] == company)
            & (df["Year"] == year)
        ]
```

**Replace with:**
```python
    @reactive.calc
    def p2_filtered_data():
        """Filter via ibis expressions, then materialize to pandas."""
        category = input.category()
        company = input.company()
        year = input.year()
        expr = tbl.filter(
            tbl["Category"] == category,
            tbl["Company"] == company,
            tbl["Year"] == year,
        )
        return expr.to_pandas()
```

---

### Detailed change 4: Add `p2_company_data()` and update chart renderers

The old code repeated `df[df["Company"] == company]` in each chart renderer.
Extract this into a single `@reactive.calc` that uses ibis:

**Add this new reactive calc right after `p2_filtered_data()`:**

```python
    @reactive.calc
    def p2_company_data():
        """All rows for the selected company (for trend charts), via ibis."""
        company = input.company()
        return tbl.filter(tbl["Company"] == company).to_pandas()
```

Then update all chart renderers that previously used `df[df["Company"] == company]`:

**Find** (in each of the 6 chart renderers: `p2_npm_chart`, `p2_roe_chart`, `p2_revenue_chart`, `p2_current_ratio_chart`, `p2_debt_equity_chart`, `p2_cash_flow_chart`):
```python
        company_data = df[df["Company"] == company]
```

**Replace with:**
```python
        company_data = p2_company_data()
```

> This also means you can remove the `company = input.company()` line from chart renderers
> that only used it for the filter — but keep it if the chart builder needs the company name.

---

### Complete chart renderer pattern (example: `p2_npm_chart`)

**Before:**
```python
    @render_altair
    def p2_npm_chart():
        company = input.company()
        company_data = df[df["Company"] == company]
        if company_data.empty:
            return empty_chart()
        return build_ratio_over_time(company_data, company, "Net Profit Margin")
```

**After:**
```python
    @render_altair
    def p2_npm_chart():
        company_data = p2_company_data()
        company = input.company()
        if company_data.empty:
            return empty_chart()
        return build_ratio_over_time(company_data, company, "Net Profit Margin")
```

Apply this same pattern to all 6 chart renderers.

---

## Step 6: Update `tests/test_data.py`

Update tests to verify the ibis/DuckDB backend.

**Replace the entire contents of `tests/test_data.py` with:**

In [ ]:
# ===== NEW tests/test_data.py =====

import ibis
import pandas as pd
from data import (
    df,
    tbl,
    con,
    ALL_SECTORS,
    METRIC_CHOICES,
    CATEGORY_COMPANIES,
    YEAR_MIN,
    YEAR_MAX,
)


def test_connection_is_duckdb():
    """Verify the ibis connection uses the DuckDB backend."""
    assert isinstance(con, ibis.backends.duckdb.Backend)


def test_tbl_is_ibis_table():
    """Verify tbl is an ibis Table expression."""
    assert isinstance(tbl, ibis.expr.types.Table)


def test_df_is_pandas_dataframe():
    """Verify df is a non-empty pandas DataFrame materialized from ibis."""
    assert isinstance(df, pd.DataFrame)
    assert len(df) > 0


def test_expected_columns_present():
    """Verify key columns required by the app exist after loading."""
    expected = {
        "Year",
        "Company",
        "Category",
        "Net Profit Margin",
        "ROE",
        "ROA",
        "ROI",
        "Revenue",
        "Net Income",
        "EBITDA",
        "Current Ratio",
        "Debt/Equity Ratio",
    }
    assert expected.issubset(set(df.columns))


def test_ibis_filter_returns_subset():
    """Verify ibis filtering works and returns fewer rows than full table."""
    filtered = tbl.filter(tbl["Category"] == "IT").to_pandas()
    assert len(filtered) > 0
    assert len(filtered) < len(df)
    assert set(filtered["Category"].unique()) == {"IT"}


def test_year_range_constants():
    """Verify YEAR_MIN and YEAR_MAX match the actual data."""
    assert YEAR_MIN == int(df["Year"].min())
    assert YEAR_MAX == int(df["Year"].max())


def test_all_sectors_sorted():
    """Verify ALL_SECTORS is a sorted list of category keys."""
    assert ALL_SECTORS == sorted(CATEGORY_COMPANIES.keys())


def test_metric_choices_non_empty():
    """Verify METRIC_CHOICES contains expected metrics."""
    assert len(METRIC_CHOICES) > 0
    assert "Net Profit Margin" in METRIC_CHOICES
    assert "Revenue" in METRIC_CHOICES


def test_category_companies_has_entries():
    """Verify CATEGORY_COMPANIES maps sectors to company lists."""
    assert len(CATEGORY_COMPANIES) > 0
    for sector, companies in CATEGORY_COMPANIES.items():
        assert isinstance(companies, list)
        assert len(companies) > 0

### What changed in tests:

| Old test | New test | Purpose |
|----------|----------|---------|
| `test_load_data_returns_dataframe` | `test_connection_is_duckdb` | Verify ibis uses DuckDB backend |
| — | `test_tbl_is_ibis_table` | Verify `tbl` is a lazy ibis Table |
| — | `test_df_is_pandas_dataframe` | Verify `df` materializes correctly |
| — | `test_ibis_filter_returns_subset` | Verify ibis filtering works end-to-end |
| — | `test_year_range_constants` | Verify `YEAR_MIN`/`YEAR_MAX` match data |
| `test_expected_columns_present` | Same | Unchanged |
| `test_all_sectors_sorted` | Same | Unchanged |
| `test_metric_choices_non_empty` | Same | Unchanged |
| `test_category_companies_has_entries` | Same | Unchanged |

**Import changes:**
- Removed: `load_data` (function no longer exists)
- Added: `tbl`, `con`, `YEAR_MIN`, `YEAR_MAX`

---

## Step 7: Verify everything works

### 7a. Run the tests

```bash
conda run -n fin-health pytest tests/ -v
```

**Expected:** All tests pass (30 passed, 3 skipped). Key new tests:
- `test_connection_is_duckdb` — PASSED
- `test_tbl_is_ibis_table` — PASSED
- `test_df_is_pandas_dataframe` — PASSED
- `test_ibis_filter_returns_subset` — PASSED
- `test_year_range_constants` — PASSED

### 7b. Run the linter

```bash
conda run -n fin-health ruff check src/ tests/ --fix && ruff format src/ tests/
```

**Expected:** `All checks passed!`

### 7c. Run the app

```bash
conda run -n fin-health shiny run src/app.py
```

**Verify manually:**
- [ ] App starts without errors
- [ ] **Sector Analysis** tab: filters work (year slider, sector dropdown, metric dropdown)
- [ ] **Company Health** tab: category/company/year filters work, KPIs and charts update
- [ ] **fin-chat** tab: querychat still responds to queries
- [ ] All charts render correctly
- [ ] Reset button on Page 1 works

---

## Step 8: Commit

```bash
git add data/processed/financial_statement.parquet src/data.py src/pages/sector.py src/pages/company.py tests/test_data.py
git commit -m "feat: migrate data loading to parquet + DuckDB via ibis"
git push origin feat/parquet-duckdb
```

---

## Architecture Summary

### Data flow (before):
```
CSV file → pd.read_csv() → pandas DataFrame (df) → pandas boolean indexing → charts
```

### Data flow (after):
```
Parquet file → ibis.duckdb.connect() → ibis Table (tbl)
                                          ├── .to_pandas() → df (for querychat)
                                          └── .filter().to_pandas() → filtered pandas (for pages)
```

### Why this matters:
- **Lazy evaluation**: ibis builds a query plan, DuckDB executes it — only matching rows are materialized
- **Parquet is columnar**: DuckDB only reads the columns it needs, faster than CSV scanning
- **Database-level filtering**: filter predicates are pushed down to DuckDB SQL, not done in Python
- **Same interface downstream**: everything after `.to_pandas()` is unchanged — charts, KPIs, tables all work as before

---

## Quick Reference: ibis vs pandas filtering

| Operation | pandas | ibis |
|-----------|--------|------|
| Single equality | `df[df["Col"] == val]` | `tbl.filter(tbl["Col"] == val)` |
| Multiple AND conditions | `df[(df["A"] == x) & (df["B"] == y)]` | `tbl.filter(tbl["A"] == x, tbl["B"] == y)` |
| IN list | `df[df["Col"].isin(lst)]` | `tbl.filter(tbl["Col"].isin(lst))` |
| Range | `df[(df["Year"] >= lo) & (df["Year"] <= hi)]` | `tbl.filter(tbl["Year"] >= lo, tbl["Year"] <= hi)` |
| Materialize | (already pandas) | `.to_pandas()` |
| Chained filter | `df = df[cond1]; df = df[cond2]` | `expr = tbl.filter(cond1); expr = expr.filter(cond2)` |

---

## Files changed checklist

- [x] `data/processed/financial_statement.parquet` — created from CSV with cleaning baked in
- [x] `src/data.py` — rewritten: `ibis.duckdb.connect()` + `con.read_parquet()`, exports `tbl`, `con`, `YEAR_MIN`, `YEAR_MAX`
- [x] `src/pages/sector.py` — import `tbl`/`YEAR_MIN`/`YEAR_MAX`, ibis filtering in `p1_filtered_data()`
- [x] `src/pages/company.py` — import `tbl`/`YEAR_MIN`/`YEAR_MAX`, ibis filtering in `p2_filtered_data()` + `p2_company_data()`
- [x] `tests/test_data.py` — new tests for ibis connection, table type, filtering, year range
- [x] `src/pages/ai_explorer.py` — **NO CHANGES** (querychat uses `df` from `data.py`, which is still pandas)